# Rule Support & Confidence Counting — DBpedia Benchmark

Dataset: `body_test_data/dbpedia.ttl` + `body_test_data/dbpedia_rules.txt`

Rules: **10 000 total** — 164 × 1 body atom, 1 240 × 2 body atoms, 8 596 × 3 body atoms

Implementations:
1. **Rewritten C++** — faithful rewrite of original Scala logic (baseline, run once — slow)
2. **Optimized CPU** — CSR layout, stack-local DFS
3. **GPU (CUDA)** — CSR + `__constant__` memory + bitmap body-size counting

In [1]:
import os, subprocess, sys, time, json, re, statistics
from pathlib import Path
from collections import defaultdict

# =============================================================================
# CHANGE THIS to wherever you uploaded / cloned the MasterThesisFinal folder.
#   Colab default : /content/drive/MyDrive/MasterThesisFinal
#   VSE Jupyter   : /home/jupyter-naav00@vse.cz/MasterThesisFinal
BASE_PATH = '/home/jupyter-naav00@vse.cz/MasterThesisFinal'
# =============================================================================

BASE = Path(BASE_PATH)
if not BASE.exists():
    raise FileNotFoundError(f'Folder not found: {BASE}\nUpdate BASE_PATH above.')
if not (BASE / 'GPU_03').exists():
    raise FileNotFoundError(f'Cannot find GPU_03 inside {BASE}')

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

def has_command(cmd):
    return subprocess.run(['which', cmd], capture_output=True).returncode == 0

HAS_NVCC = has_command('nvcc')
HAS_GPU  = False
if HAS_NVCC:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                       capture_output=True, text=True)
    HAS_GPU = r.returncode == 0 and r.stdout.strip() != ''

print(f'Environment : {"Google Colab" if IS_COLAB else "JupyterHub / local"}')
print(f'nvcc found  : {HAS_NVCC}')
print(f'GPU found   : {HAS_GPU}')
if HAS_GPU:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap',
                          '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

DIR_REWRITTEN = BASE / 'Rewritten_CPP'
DIR_CPU       = BASE / 'CPU_01_nonparallel'
DIR_GPU       = BASE / 'GPU_03'
DATA_DIR      = BASE / 'body_test_data'

TTL_FILE  = DATA_DIR / 'dbpedia.ttl'
RULE_FILE = DATA_DIR / 'dbpedia_rules.txt'

RULE_FILE_JSON = DATA_DIR / 'dbpedia_rules.json'
print(f'\nBase dir  : {BASE}')
print(f'TTL file  : {TTL_FILE}  (exists={TTL_FILE.exists()})')
print(f'Rules txt : {RULE_FILE} (exists={RULE_FILE.exists()})')
print(f'Rules json: {RULE_FILE_JSON} (exists={RULE_FILE_JSON.exists()})')

Environment : JupyterHub / local
nvcc found  : True
GPU found   : True
NVIDIA H200 NVL, 9.0
NVIDIA H200 NVL, 9.0
NVIDIA H200 NVL, 9.0
NVIDIA H200 NVL, 9.0

Base dir  : /home/jupyter-naav00@vse.cz/MasterThesisFinal
TTL file  : /home/jupyter-naav00@vse.cz/MasterThesisFinal/body_test_data/dbpedia.ttl  (exists=True)
Rules txt : /home/jupyter-naav00@vse.cz/MasterThesisFinal/body_test_data/dbpedia_rules.txt (exists=True)
Rules json: /home/jupyter-naav00@vse.cz/MasterThesisFinal/body_test_data/dbpedia_rules.json (exists=True)


In [2]:
# -- Helper: run a shell command and print output -------------------------------
def run(cmd, cwd=None, check=True):
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, cwd=cwd or BASE,
                            capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode})')
    return result

# -- Helper: run a benchmark binary N times, return (min_ms, avg_ms, last_stdout)
def benchmark(cmd, cwd=None, runs=3):
    times = []
    last_stdout = ''
    for i in range(runs):
        t0 = time.perf_counter()
        r  = subprocess.run(cmd, shell=True, cwd=cwd or BASE,
                            capture_output=True, text=True)
        t1 = time.perf_counter()
        if r.returncode != 0:
            raise RuntimeError(f'Benchmark failed: {r.stderr}')
        times.append((t1 - t0) * 1000)
        last_stdout = r.stdout
    return min(times), sum(times)/len(times), last_stdout

# -- Helper: extract total wall time from stdout --------------------------------
def extract_total_ms(stdout):
    for pattern in [r'Total wall time:\s+([0-9.]+)\s*ms',
                    r'Total time:\s+([0-9.]+)\s*ms']:
        m = re.search(pattern, stdout)
        if m: return float(m.group(1))
    return None

# -- Helper: count body atoms from a rule text string -------------------------
def count_body_atoms(rule_text):
    body = rule_text.split('=>')[0]
    return body.count('^') + 1

# -- Helper: parse Rewritten C++ stdout into per-rule records ------------------
def parse_rewritten_output(stdout):
    results = []
    lines = stdout.split('\n')
    i = 0
    while i < len(lines):
        m = re.match(r'Rule \d+: (.+)', lines[i])
        if m:
            rule_text = m.group(1)
            j = i + 1
            while j < len(lines) and 'support=' not in lines[j]:
                j += 1
            if j < len(lines):
                sm = re.search(r'supportMs=([0-9.]+)', lines[j])
                cm = re.search(r'confidenceMs=([0-9.]+)', lines[j])
                if sm and cm:
                    results.append({
                        'body_atoms': count_body_atoms(rule_text),
                        'supportMs':    float(sm.group(1)),
                        'confidenceMs': float(cm.group(1))
                    })
            i = j + 1
        else:
            i += 1
    return results

# -- Helper: parse CPU --json stdout into per-rule records ---------------------
def parse_cpu_json(stdout):
    data = json.loads(stdout)
    return [{
        'body_atoms':    count_body_atoms(r['rule']),
        'supportMs':     r['supportMs'],
        'confidenceMs':  r['confidenceMs']
    } for r in data]


# -- Helper: parse GPU bench_gpu stdout into per-rule records ------------------
def parse_gpu_output(stdout):
    results = []
    for line in stdout.splitlines():
        # Format: Rule N: <rule text> | ... SupportTime: X ms, ConfidenceTime: Y ms
        if not line.startswith('Rule '):
            continue
        pipe = line.find('|')
        if pipe == -1:
            continue
        rule_text = line[line.index(':') + 1:pipe].strip()
        sm = re.search(r'SupportTime:\s*([0-9.]+)', line)
        cm = re.search(r'ConfidenceTime:\s*([0-9.]+)', line)
        if sm and cm:
            results.append({
                'body_atoms': count_body_atoms(rule_text),
                'supportMs': float(sm.group(1)),
                'confidenceMs': float(cm.group(1)),
            })
    return results

# -- Helper: print per-body-size statistics ------------------------------------
def print_body_stats(records, label):
    groups = defaultdict(list)
    for r in records:
        groups[r['body_atoms']].append(r)
    total_sup  = sum(r['supportMs']    for r in records)
    total_conf = sum(r['confidenceMs'] for r in records)
    print(f'\n=== {label} — Statistics by body size ===')
    print(f'Total support time:    {total_sup/1000:.2f} s')
    print(f'Total confidence time: {total_conf/1000:.2f} s')
    for n in sorted(groups.keys()):
        g = groups[n]
        sup  = [r['supportMs']    for r in g]
        conf = [r['confidenceMs'] for r in g]
        print(f'\nBody size {n} ({len(g)} rules):')
        print(f'  Support:    avg={statistics.mean(sup):.1f} ms  median={statistics.median(sup):.1f} ms  '
              f'min={min(sup):.1f} ms  max={max(sup):.1f} ms')
        print(f'  Confidence: avg={statistics.mean(conf):.1f} ms  median={statistics.median(conf):.1f} ms  '
              f'min={min(conf):.1f} ms  max={max(conf):.1f} ms')


import psutil, threading

def get_vram_mb():
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
        capture_output=True, text=True)
    if r.returncode != 0: return 0
    return int(r.stdout.strip().split('\n')[0])

def measure_peak_memory(cmd, cwd=None, interval=0.5):
    baseline_ram  = psutil.virtual_memory().used / (1024*1024)
    baseline_vram = get_vram_mb()
    peak_ram  = [baseline_ram]
    peak_vram = [baseline_vram]
    stop_flag = [False]
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or BASE,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    def sampler():
        while not stop_flag[0]:
            peak_ram[0]  = max(peak_ram[0],  psutil.virtual_memory().used / (1024*1024))
            peak_vram[0] = max(peak_vram[0], get_vram_mb())
            time.sleep(interval)
    t = threading.Thread(target=sampler, daemon=True)
    t0 = time.perf_counter()
    t.start()
    for line in proc.stdout:
        pass  # drain output silently
    proc.wait()
    stop_flag[0] = True
    t.join()
    wall_ms = (time.perf_counter() - t0) * 1000
    return peak_ram[0] - baseline_ram, peak_vram[0] - baseline_vram, wall_ms

print('Helpers ready.')

Helpers ready.


---
## Step 1 — Compile all implementations

In [3]:
print('Compiling Rewritten_CPP ...')
run(
    'g++ -O3 -std=c++17 '
    'main.cpp FinalRule.cpp RdfIndexes.cpp RuleParser.cpp SupportCounting.cpp '
    '-o rdf_rules_rewritten',
    cwd=DIR_REWRITTEN
)
print('OK — binary: Rewritten_CPP/rdf_rules_rewritten')

Compiling Rewritten_CPP ...
$ g++ -O3 -std=c++17 main.cpp FinalRule.cpp RdfIndexes.cpp RuleParser.cpp SupportCounting.cpp -o rdf_rules_rewritten
OK — binary: Rewritten_CPP/rdf_rules_rewritten


In [4]:
print('Compiling CPU_01_nonparallel ...')
run(
    'g++ -O3 -std=c++17 '
    'main.cpp FinalRule.cpp RdfIndexes.cpp RuleParser.cpp SupportCounting.cpp '
    '-o rdf_rules_test',
    cwd=DIR_CPU
)
print('OK — binary: CPU_01_nonparallel/rdf_rules_test')

Compiling CPU_01_nonparallel ...
$ g++ -O3 -std=c++17 main.cpp FinalRule.cpp RdfIndexes.cpp RuleParser.cpp SupportCounting.cpp -o rdf_rules_test
OK — binary: CPU_01_nonparallel/rdf_rules_test


In [5]:
if not HAS_NVCC:
    print('nvcc not found — skipping GPU compilation.')
else:
    print('Compiling GPU_03 benchmark binary ...')
    run(
        'nvcc -O3 -std=c++17 -Wno-deprecated-gpu-targets '
        'bench_gpu.cu FinalRule.cpp RdfIndexes.cpp RuleParser.cpp '
        '-o bench_gpu',
        cwd=DIR_GPU
    )
    print('OK — binary: GPU_03/bench_gpu')

    print('\nCompiling GPU_03 shared library ...')
    run(
        'nvcc -O3 -std=c++17 -Wno-deprecated-gpu-targets -shared -Xcompiler -fPIC '
        'gpu_rules_lib.cu FinalRule.cpp RdfIndexes.cpp RuleParser.cpp '
        '-o libgpu_rules.so',
        cwd=DIR_GPU
    )
    print('OK — library: GPU_03/libgpu_rules.so')

Compiling GPU_03 benchmark binary ...
$ nvcc -O3 -std=c++17 -Wno-deprecated-gpu-targets bench_gpu.cu FinalRule.cpp RdfIndexes.cpp RuleParser.cpp -o bench_gpu
OK — binary: GPU_03/bench_gpu

Compiling GPU_03 shared library ...
$ nvcc -O3 -std=c++17 -Wno-deprecated-gpu-targets -shared -Xcompiler -fPIC gpu_rules_lib.cu FinalRule.cpp RdfIndexes.cpp RuleParser.cpp -o libgpu_rules.so
OK — library: GPU_03/libgpu_rules.so


---
## Step 2 — Run benchmarks

**Note**: Rewritten C++ is run **once** (it is the slow baseline — 10 000 rules may take a long time).  
Optimized CPU and GPU are run **3 times**; we report the fastest run.

In [6]:
print('Running Rewritten_CPP benchmark (1 run — this may take a while) ...')
cmd_rewritten = f'{DIR_REWRITTEN}/rdf_rules_rewritten {TTL_FILE} {RULE_FILE_JSON}'
min_ms_r, avg_ms_r, out_r = benchmark(cmd_rewritten, runs=1)
total_r = extract_total_ms(out_r)

print(out_r[-3000:])  # print last 3000 chars (summary section)
print(f'Wall time: {min_ms_r:.0f} ms  (reported total={total_r} ms)')

records_r = parse_rewritten_output(out_r)
print_body_stats(records_r, 'Rewritten C++')

Running Rewritten_CPP benchmark (1 run — this may take a while) ...
6582  bodySize=54  confidence=0.0555556  supportMs=0  confidenceMs=0
Rule 9992: ( ?0 http://dbpedia.org/ontology/deathPlace ?1 ) ^ ( ?0 http://dbpedia.org/ontology/birthPlace ?2 ) ^ ( ?1 http://dbpedia.org/ontology/country ?3 ) => ( ?3 http://dbpedia.org/ontology/capital ?2 )
  support=3  headCoverage=0.0126582  bodySize=230  confidence=0.0130435  supportMs=0  confidenceMs=0
Rule 9993: ( ?0 http://dbpedia.org/ontology/timeZone ?1 ) ^ ( ?2 http://dbpedia.org/ontology/timeZone ?1 ) ^ ( ?0 http://dbpedia.org/ontology/country ?3 ) => ( ?3 http://dbpedia.org/ontology/capital ?2 )
  support=10  headCoverage=0.0421941  bodySize=616  confidence=0.0162338  supportMs=0  confidenceMs=0
Rule 9994: ( ?0 http://dbpedia.org/ontology/country ?1 ) ^ ( ?2 http://dbpedia.org/ontology/country ?1 ) ^ ( ?0 http://dbpedia.org/ontology/country ?3 ) => ( ?3 http://dbpedia.org/ontology/capital ?2 )
  support=3  headCoverage=0.0126582  bodySize=

In [7]:
print('Running CPU_01_nonparallel benchmark (3 runs) ...')
cmd_cpu = f'{DIR_CPU}/rdf_rules_test {TTL_FILE} {RULE_FILE_JSON}'
min_ms_c, avg_ms_c, out_c = benchmark(cmd_cpu, runs=3)
total_c = extract_total_ms(out_c)

print(out_c[-3000:])
print(f'Wall time: min={min_ms_c:.0f} ms  avg={avg_ms_c:.0f} ms  (reported total={total_c} ms)')

# Run once more with --json to get per-rule data for body-size breakdown
print('\nCollecting per-rule data (--json) ...')
r_json = subprocess.run(
    f'{DIR_CPU}/rdf_rules_test {TTL_FILE} {RULE_FILE_JSON} --json',
    shell=True, capture_output=True, text=True
)
records_c = parse_cpu_json(r_json.stdout)
print_body_stats(records_c, 'Optimized CPU')

Running CPU_01_nonparallel benchmark (3 runs) ...
p://dbpedia.org/ontology/birthPlace ?2 ) ^ ( ?1 http://dbpedia.org/ontology/country ?3 ) => ( ?3 http://dbpedia.org/ontology/capital ?2 )
  support=3  headCoverage=0.0126582  bodySize=230  confidence=0.0130435  supportMs=0.05178  confidenceMs=0.116676
Rule 9993: ( ?0 http://dbpedia.org/ontology/timeZone ?1 ) ^ ( ?2 http://dbpedia.org/ontology/timeZone ?1 ) ^ ( ?0 http://dbpedia.org/ontology/country ?3 ) => ( ?3 http://dbpedia.org/ontology/capital ?2 )
  support=10  headCoverage=0.0421941  bodySize=616  confidence=0.0162338  supportMs=0.048437  confidenceMs=0.163172
Rule 9994: ( ?0 http://dbpedia.org/ontology/country ?1 ) ^ ( ?2 http://dbpedia.org/ontology/country ?1 ) ^ ( ?0 http://dbpedia.org/ontology/country ?3 ) => ( ?3 http://dbpedia.org/ontology/capital ?2 )
  support=3  headCoverage=0.0126582  bodySize=6080  confidence=0.000493421  supportMs=0.03509  confidenceMs=0.523406
Rule 9995: ( ?0 http://dbpedia.org/ontology/philosophicalSc

In [8]:
min_ms_g = avg_ms_g = total_g = None
records_g = []

if not HAS_GPU:
    print('No GPU available — skipping GPU benchmark.')
else:
    print('Running GPU_03 benchmark (3 runs) ...')
    cmd_gpu = f'{DIR_GPU}/bench_gpu {TTL_FILE} {RULE_FILE_JSON}'
    min_ms_g, avg_ms_g, out_g = benchmark(cmd_gpu, runs=3)
    total_g = extract_total_ms(out_g)
    records_g = parse_gpu_output(out_g)

    # Only print from the timing summary onward — skip per-rule lines
    summary_start = out_g.find('=== Timing summary ===')
    if summary_start != -1:
        print(out_g[summary_start:])
    else:
        print(out_g[-3000:])

    print(f'Wall time: min={min_ms_g:.0f} ms  avg={avg_ms_g:.0f} ms  (reported total={total_g} ms)')

Running GPU_03 benchmark (3 runs) ...
=== Timing summary ===
Indexing time:          321.966 ms
CSR build+upload time:  1916.24 ms
Rule parsing time:      110.463 ms
Support counting time:  658.724 ms
Confidence counting time: 2740.57 ms
Total time:             5748.69 ms
Warp-coop rules:        292  (threshold maxfan > 64)

=== Per-rule SUPPORT statistics (10000 rules) ===
Average: 0.0294506 ms
Median:  0.011488 ms
Std Dev: 0.0750354 ms
Min:     0.006816 ms
Max:     1.61776 ms
P95:     0.126656 ms

=== Per-rule CONFIDENCE statistics (10000 rules) ===
Average: 0.240461 ms
Median:  0.10112 ms
Std Dev: 0.691848 ms
Min:     0 ms
Max:     24.3736 ms
P95:     0.798784 ms

=== SUPPORT statistics by body size ===
Body size 1 (164 rules): avg=0.00773971 ms, median=0.007472 ms, min=0.006816 ms, max=0.046592 ms, P95=0.008224 ms
Body size 2 (1240 rules): avg=0.0109303 ms, median=0.009568 ms, min=0.007104 ms, max=0.048736 ms, P95=0.019616 ms
Body size 3 (8596 rules): avg=0.0325364 ms, median=0.012

---
## Step 3 — Rule-computation speedup table

In [9]:
# If the GPU cell was already run before this notebook fix, recover records_g from out_g.
if 'records_g' not in globals():
    records_g = parse_gpu_output(out_g) if 'out_g' in globals() else []

def total_rule_compute_ms(records):
    return sum(r['supportMs'] + r['confidenceMs'] for r in records)

compute_ms_r = total_rule_compute_ms(records_r)
compute_ms_c = total_rule_compute_ms(records_c)
compute_ms_g = total_rule_compute_ms(records_g) if records_g else None

rows = [
    {'Implementation': 'Rewritten C++',
     'Rule compute time (ms)': f'{compute_ms_r:.3f}',
     'Speedup vs Rewritten': '1.0x'},
    {'Implementation': 'Optimized CPU',
     'Rule compute time (ms)': f'{compute_ms_c:.3f}',
     'Speedup vs Rewritten': f'{compute_ms_r/compute_ms_c:.2f}x' if compute_ms_c else 'N/A'},
]
if compute_ms_g is not None:
    rows.append({
        'Implementation': 'GPU (CUDA)',
        'Rule compute time (ms)': f'{compute_ms_g:.3f}',
        'Speedup vs Rewritten': f'{compute_ms_r/compute_ms_g:.2f}x' if compute_ms_g else 'N/A'
    })
else:
    rows.append({
        'Implementation': 'GPU (CUDA)',
        'Rule compute time (ms)': 'N/A (no GPU)',
        'Speedup vs Rewritten': 'N/A'
    })

print('Dataset: DBpedia  |  Rules: 10 000  (164×1atom + 1240×2atom + 8596×3atom)')
print('Speedup is based only on rule computation: sum(supportMs + confidenceMs).')
print('Wall time / initialization / graph loading / indexing / CUDA startup are excluded.\n')
header = f'{"Implementation":<22} {"Rule compute (ms)":>20} {"Speedup vs Rewritten":>22}'
print(header)
print('-' * len(header))
for r in rows:
    print(f'{r["Implementation"]:<22} {r["Rule compute time (ms)"]:>20} {r["Speedup vs Rewritten"]:>22}')


Dataset: DBpedia  |  Rules: 10 000  (164×1atom + 1240×2atom + 8596×3atom)
Speedup is based only on rule computation: sum(supportMs + confidenceMs).
Wall time / initialization / graph loading / indexing / CUDA startup are excluded.

Implementation            Rule compute (ms)   Speedup vs Rewritten
------------------------------------------------------------------
Rewritten C++                     53148.000                   1.0x
Optimized CPU                     43078.612                  1.23x
GPU (CUDA)                         2699.120                 19.69x


---
## Memory Usage — Peak RAM and VRAM per implementation

In [10]:
print('Measuring peak memory usage (this re-runs each implementation silently) ...\n')

ram_r, vram_r, _ = measure_peak_memory(f'{DIR_REWRITTEN}/rdf_rules_rewritten {TTL_FILE} {RULE_FILE_JSON}')
print(f'Rewritten C++  — RAM: {ram_r:>8,.0f} MB   VRAM: {vram_r:>6,.0f} MB')

ram_c, vram_c, _ = measure_peak_memory(f'{DIR_CPU}/rdf_rules_test {TTL_FILE} {RULE_FILE_JSON}')
print(f'Optimized CPU  — RAM: {ram_c:>8,.0f} MB   VRAM: {vram_c:>6,.0f} MB')

if HAS_GPU:
    ram_g, vram_g, _ = measure_peak_memory(f'{DIR_GPU}/bench_gpu {TTL_FILE} {RULE_FILE_JSON}')
    print(f'GPU (CUDA)     — RAM: {ram_g:>8,.0f} MB   VRAM: {vram_g:>6,.0f} MB')
else:
    print('GPU (CUDA)     — skipped (no GPU)')

Measuring peak memory usage (this re-runs each implementation silently) ...

Rewritten C++  — RAM:      120 MB   VRAM:      0 MB
Optimized CPU  — RAM:      113 MB   VRAM:      0 MB
GPU (CUDA)     — RAM:      365 MB   VRAM:  1,669 MB
